In [8]:
QUERY_MS = """
(
LRT3
OR "LRT Shah Alam"
OR "Shah Alam Line"
)
-is:retweet
"""

In [9]:
import os
import requests
import pandas as pd

BEARER_TOKEN = os.getenv("BEARER_TOKEN")

headers = {
    "Authorization": f"Bearer {BEARER_TOKEN}"
}

url = "https://api.x.com/2/tweets/search/recent"

params = {
    "query": QUERY_MS,
    "max_results": 100,
    "tweet.fields": "created_at,lang,public_metrics",
}

all_tweets = []

while True:

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    data = response.json()

    if "data" in data:
        all_tweets.extend(data["data"])

    next_token = data.get("meta", {}).get("next_token")

    if not next_token:
        break

    params["next_token"] = next_token

df = pd.DataFrame(all_tweets)

print(df.head())

Empty DataFrame
Columns: []
Index: []


In [19]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

BEARER_TOKEN = os.getenv("BEARER_TOKEN")

headers = {
    "Authorization": f"Bearer {BEARER_TOKEN}"
}


# Broad RapidKL-related query
QUERY = """
(
LRT3 OR
"LRT Shah Alam" OR
"Shah Alam Line" 
)
-is:retweet
"""

url = "https://api.x.com/2/tweets/search/recent"

params = {
    "query": QUERY,
    "max_results": 100,
    "tweet.fields": "created_at,lang,public_metrics,author_id",
}

all_tweets = []

while True:

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    print(f"Status Code: {response.status_code}")

    if response.status_code != 200:
        print(response.json())
        break

    data = response.json()

    print("Tweets returned:", data.get("meta", {}).get("result_count", 0))

    if "data" in data:
        all_tweets.extend(data["data"])

    next_token = data.get("meta", {}).get("next_token")

    if not next_token:
        break

    params["next_token"] = next_token

# Convert to DataFrame
df = pd.DataFrame(all_tweets)

print(f"\nTotal Tweets Collected: {len(df)}")

if not df.empty:

    print("\nLanguage Distribution:")
    print(df["lang"].value_counts())

    df.to_csv(
        "rapidkl_tweets.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nSaved to rapidkl_tweets.csv")

else:
    print("\nNo tweets found.")

Status Code: 200
Tweets returned: 98
Status Code: 200
Tweets returned: 100
Status Code: 200
Tweets returned: 44

Total Tweets Collected: 242

Language Distribution:
lang
in     149
en      81
ja       5
zh       4
uk       1
tl       1
und      1
Name: count, dtype: int64

Saved to rapidkl_tweets.csv


In [30]:
df = pd.read_csv("rapidkl_tweets.csv")

In [31]:
df = df[['created_at','lang','text']]
df

,created_at,lang,text
0,2026-07-20T01:11:29.000Z,en,"@anthonyloke @AlexNantaLinggi YB, pls deliver ..."
1,2026-07-19T22:28:06.000Z,in,Rail Frequency Updates (7.00 AM - 9.30 AM)\n\n...
2,2026-07-19T17:38:37.000Z,en,LRT3 first shovel was in 2015 and finally comp...
3,2026-07-19T15:08:00.000Z,zh,馬來西亞吉隆坡LRT3輕快鐵通車首週，就發生列車行駛途中突然停駛，並伴隨異響與火花。所幸事件...
4,2026-07-19T13:39:33.000Z,zh,FB电脑版瘫痪..\n应该是大家集体罢工，\n搭LRT3去巴生吃肉骨茶了..\n\n嗯.. ...
...,...,...,...
237,2026-07-13T05:33:06.000Z,in,Sultan Selangor tinjau operasi LRT3 https://t....
238,2026-07-13T05:31:27.000Z,in,Alangkah baiknya kalau LRT3 siap seperti peran...
239,2026-07-13T05:00:54.000Z,in,Sultan Selangor tinjau operasi LRT3 laluan Sha...
240,2026-07-13T04:22:17.000Z,in,Wartawan pertahan artikel LRT3 sebagai komen a...


In [32]:
df['lang'].unique()

<StringArray>
['en', 'in', 'zh', 'uk', 'ja', 'tl', 'und']
Length: 7, dtype: str

In [33]:
dfnew = df[~df["lang"].isin(["zh", "ja", "tl"])]
dfnew

,created_at,lang,text
0,2026-07-20T01:11:29.000Z,en,"@anthonyloke @AlexNantaLinggi YB, pls deliver ..."
1,2026-07-19T22:28:06.000Z,in,Rail Frequency Updates (7.00 AM - 9.30 AM)\n\n...
2,2026-07-19T17:38:37.000Z,en,LRT3 first shovel was in 2015 and finally comp...
5,2026-07-19T12:48:16.000Z,uk,Минулої п'ятниці я зробив багато фотографій вз...
6,2026-07-19T12:47:33.000Z,en,"Last Friday, I took a bunch of pics along the ..."
...,...,...,...
237,2026-07-13T05:33:06.000Z,in,Sultan Selangor tinjau operasi LRT3 https://t....
238,2026-07-13T05:31:27.000Z,in,Alangkah baiknya kalau LRT3 siap seperti peran...
239,2026-07-13T05:00:54.000Z,in,Sultan Selangor tinjau operasi LRT3 laluan Sha...
240,2026-07-13T04:22:17.000Z,in,Wartawan pertahan artikel LRT3 sebagai komen a...


In [34]:
dfnew.to_csv("lrt3-tweets.csv",index=False)

In [6]:
import pandas as pd 
dfnew = pd.read_csv("lrt3-tweets.csv")

In [7]:
import shutil
import os

path = r"C:\Users\nasuha.imanina\.cache\huggingface\hub\models--cardiffnlp--twitter-xlm-roberta-base-sentiment"

if os.path.exists(path):
    shutil.rmtree(path)

print("Deleted corrupted cache")

Deleted corrupted cache


In [8]:
import sentencepiece

print(sentencepiece.__version__)

0.2.2


In [27]:
from transformers import pipeline
import pandas as pd

# Load model
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis"
)


def categorize_sentiment(text):
    try:
        result = sentiment_analyzer(str(text))[0]

        label = result['label'].lower()
        score = result['score']

        # Map sentiment
        if label in ['very positive', 'positive']:
            category = 'Positive'

        elif label == 'neutral':
            category = 'Neutral'

        elif label in ['negative', 'very negative']:
            category = 'Negative'

        else:
            category = 'Unknown'

        return pd.Series([label, score, category])

    except Exception as e:
        print(f"Error: {e}")
        return pd.Series([None, None, None])


# Apply sentiment analysis
dfnew[['sentiment_label', 'confidence', 'sentiment_category']] = (
    dfnew['text']
    .fillna('')
    .apply(categorize_sentiment)
)

print(dfnew.head())

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11557.01it/s]


                 created_at lang  \
0  2026-07-20T01:11:29.000Z   en   
1  2026-07-19T22:28:06.000Z   in   
2  2026-07-19T17:38:37.000Z   en   
3  2026-07-19T12:48:16.000Z   uk   
4  2026-07-19T12:47:33.000Z   en   

                                                text sentiment_label  \
0  @anthonyloke @AlexNantaLinggi YB, pls deliver ...        negative   
1  Rail Frequency Updates (7.00 AM - 9.30 AM)\n\n...         neutral   
2  LRT3 first shovel was in 2015 and finally comp...         neutral   
3  Минулої п'ятниці я зробив багато фотографій вз...         neutral   
4  Last Friday, I took a bunch of pics along the ...         neutral   

   confidence sentiment_category  
0    0.766959           Negative  
1    0.936337            Neutral  
2    0.433166            Neutral  
3    0.847012            Neutral  
4    0.747381            Neutral  


In [ ]:
from transformers import pipeline
import pandas as pd

# Load model
# sentiment_analyzer = pipeline(
#     "sentiment-analysis",
#     model="w11wo/indonesian-roberta-base-sentiment-classifier"
# )

# sentiment_analyzer = pipeline(
#     "sentiment-analysis",
#     model="finiteautomata/bertweet-base-sentiment-analysis"
# )

# sentiment_analyzer = pipeline(
#     "text-classification",
#     model="rmtariq/ft-Malay-bert"
# )
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis"
)

def categorize_sentiment(text):
    try:
        result = sentiment_analyzer(str(text))[0]

        label = result['label'].lower()
        score = result['score']

        #    # Map sentiment
        # if label == 'pos':
        #     category = 'Positive'
        # elif label == 'neu':
        #     category = 'Neutral'
        # elif label == 'neg':
        #     category = 'Negative'
        # else:
        #     category = 'Unknown'

        # Map sentiment
        if label == 'positive':
            category = 'Positive'
        elif label == 'neutral':
            category = 'Neutral'
        elif label == 'negative':
            category = 'Negative'
        else:
            category = 'Unknown'

        return pd.Series([label, score, category])

    except Exception as e:
        print(f"Error: {e}")
        return pd.Series([None, None, None])


# Apply sentiment analysis
dfnew[['sentiment_label', 'confidence', 'sentiment_category']] = (
    dfnew['text']
    .fillna('')
    .apply(categorize_sentiment)
)

print(dfnew.head())

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5930.92it/s]


                 created_at lang  \
0  2026-07-20T01:11:29.000Z   en   
1  2026-07-19T22:28:06.000Z   in   
2  2026-07-19T17:38:37.000Z   en   
3  2026-07-19T12:48:16.000Z   uk   
4  2026-07-19T12:47:33.000Z   en   

                                                text sentiment_label  \
0  @anthonyloke @AlexNantaLinggi YB, pls deliver ...        negative   
1  Rail Frequency Updates (7.00 AM - 9.30 AM)\n\n...         neutral   
2  LRT3 first shovel was in 2015 and finally comp...         neutral   
3  Минулої п'ятниці я зробив багато фотографій вз...         neutral   
4  Last Friday, I took a bunch of pics along the ...         neutral   

   confidence sentiment_category  
0    0.766959           Negative  
1    0.936337            Neutral  
2    0.433166            Neutral  
3    0.847012            Neutral  
4    0.747381            Neutral  


In [28]:
dfnew

,created_at,lang,text,sentiment_label,confidence,sentiment_category
0,2026-07-20T01:11:29.000Z,en,"@anthonyloke @AlexNantaLinggi YB, pls deliver ...",negative,0.766959,Negative
1,2026-07-19T22:28:06.000Z,in,Rail Frequency Updates (7.00 AM - 9.30 AM)\n\n...,neutral,0.936337,Neutral
2,2026-07-19T17:38:37.000Z,en,LRT3 first shovel was in 2015 and finally comp...,neutral,0.433166,Neutral
3,2026-07-19T12:48:16.000Z,uk,Минулої п'ятниці я зробив багато фотографій вз...,neutral,0.847012,Neutral
4,2026-07-19T12:47:33.000Z,en,"Last Friday, I took a bunch of pics along the ...",neutral,0.747381,Neutral
...,...,...,...,...,...,...
227,2026-07-13T05:33:06.000Z,in,Sultan Selangor tinjau operasi LRT3 https://t....,neutral,0.908687,Neutral
228,2026-07-13T05:31:27.000Z,in,Alangkah baiknya kalau LRT3 siap seperti peran...,neutral,0.852225,Neutral
229,2026-07-13T05:00:54.000Z,in,Sultan Selangor tinjau operasi LRT3 laluan Sha...,neutral,0.917516,Neutral
230,2026-07-13T04:22:17.000Z,in,Wartawan pertahan artikel LRT3 sebagai komen a...,neutral,0.909631,Neutral


In [29]:
dfnew.to_csv("lrt3-sentiment.csv",index=False)